# Phase 8 — Gradio Application Integration

**Roadmap reference:** `08_Gradio_Integration.ipynb`
This notebook imports reusable code from `src/contractiq` only
(`app_service.py` + `ui.py`) — it does not depend on hidden execution
order from earlier notebooks, per the roadmap's explicit requirement.

**Design source:** `PWC_CONTRACTIQ_PROJECT_IK/design/index.html` — a 9-section
sidebar-nav workspace (Overview, Contract Library, Risk Review, Ask
ContractIQ, Document Sources, Relationships, Review Tasks, Reports,
Activity & Traces). The same 9 sections are rebuilt here as native Gradio
(`gr.Sidebar` + `gr.Tabs`), each wired to `app_service.py` — which wraps
ALREADY-VERIFIED Phase 0-7 code, not a reimplementation.

**Honest constraint:** this is a native Gradio layout, not a pixel clone
of the mockup's custom hand-written HTML/CSS/JS (Gradio can't render
arbitrary custom markup). What's actually reused is the mockup's
*information architecture* — same 9 sections, same left-sidebar navigation
pattern, a teal color accent approximating the mockup's palette.

**Per-section backend mapping (what's reused vs. genuinely new):**

| Section | Backend |
|---|---|
| Overview | Phase 0 manifest + Phase 5 assessments + Phase 6 graph + Phase 7 eval report |
| Contract Library | Phase 0 manifest (`list_documents`) |
| Risk Review | Phase 5 `Assessment` (`get_risk_review`) |
| Ask ContractIQ | Phase 3 `search()` (`ask`) |
| Document Sources | Phase 3 ChromaDB (`get_document_chunks` — one new facade method) |
| Relationships | Phase 6 `ContractKnowledgeGraph` (reused directly, not rebuilt) |
| Review Tasks | **Genuinely new** — local JSON-backed CRUD, no earlier phase built this |
| Reports | Phase 7 `generate_document_report()` (reused directly) |
| Activity & Traces | Phase 7's real `phase7_llm_calls.jsonl` log (not a fabricated "demo" notice) |

**Completion gate:** the app launches from this notebook using only
`src/contractiq` imports; every section either shows real data from an
earlier phase or is honestly labeled as new.

## 1. Load configuration

In [1]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq import app_service as svc
from src.contractiq.ui import build_app

print(f"Manifest documents: {len(svc.list_documents())}")
print(f"Assessed documents: {len(svc.list_assessed_doc_ids())}")
print(f"Langfuse host: {config.LANGFUSE_HOST}")

e:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\.contract\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Manifest documents: 36
Assessed documents: 15
Langfuse host: https://us.cloud.langfuse.com


## 2. Sanity-check the service facade before launching the UI

Every number below comes from a real saved artifact (Phase 0 manifest,
Phase 5 assessments, Phase 6 graph, Phase 7 evaluation report) — nothing
here is placeholder data for the UI to display.

In [2]:
overview = svc.get_overview()
print(f"Documents: {overview['total_documents']} total, {overview['assessed_documents']} assessed")
print(f"Risk distribution: {overview['risk_distribution']}")
print(f"Knowledge graph: {overview['graph_nodes']} nodes, {overview['graph_edges']} edges")
print(f"Citation validity: {overview['citation_validity_rate']}")
print(f"Retrieval relevance: {overview['retrieval_relevance_rate']}")

Documents: 36 total, 15 assessed
Risk distribution: {'MEDIUM': 8, 'HIGH': 7}
Knowledge graph: 50 nodes, 68 edges
Citation validity: 0.946
Retrieval relevance: 1.0


## 3. Build and launch the app

`build_app()` assembles all 9 sections; `.launch()` applies the teal theme
and starts the local Gradio server. `prevent_thread_lock=True` so this
notebook cell returns control instead of blocking forever — the server
keeps running in the background until the kernel is stopped.

In [3]:
from src.contractiq.ui import launch_app

# launch_app() applies the teal theme + custom CSS + Google-Fonts <head> link.
# Gradio 6 moved theme/css/head from the Blocks constructor to launch().
launch_app(
    server_name="127.0.0.1",
    server_port=7860,
    inbrowser=True,
    share=False,
    prevent_thread_lock=True,
)
print("ContractIQ is running at http://127.0.0.1:7860")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ContractIQ is running at http://127.0.0.1:7860


## 4. FastAPI deployment surface

`contractiq/api.py` exposes the same `app_service.py` facade over HTTP, so
the API and the UI can never disagree about what the system found. Every
endpoint is a thin adapter — no analysis logic lives in the API layer.

Run it standalone:

```bash
uvicorn contractiq.api:app --reload --port 8000   # docs at /docs
```

The cell below exercises the endpoints in-process with FastAPI's
`TestClient` — no server needed, and no mocking: these hit the real saved
artifacts.

In [ ]:
from fastapi.testclient import TestClient

from src.contractiq.api import app as api_app

api = TestClient(api_app)

print("GET  /health                 ", api.get("/health").json())
print("GET  /api/documents          ", len(api.get("/api/documents").json()), "documents")
print("GET  /api/assessments        ", len(api.get("/api/assessments").json()), "assessed")

doc_id = api.get("/api/assessments").json()[0]
assessment = api.get(f"/api/assessments/{doc_id}").json()
print(f"GET  /api/assessments/{doc_id}  overall_risk={assessment['overall_risk']}, "
      f"{len(assessment['findings'])} findings")

# Citation drill-down: the endpoint that proves where a finding came from.
finding = assessment["findings"][0]
chunk_id = finding["evidence_ids"][0]
citation = api.get(f"/api/citations/{doc_id}/{chunk_id}",
                   params={"quote": finding["evidence_excerpts"][0]}).json()
print(f"\nGET  /api/citations/{doc_id}/{chunk_id}")
print(f"  file: {citation['filename']}  range: {citation['source_location_range']}")
print(f"  quote located in source: {citation['quote_located']}")
for para in citation["paragraphs"]:
    if para["is_quote_source"]:
        print(f"  --> {para['locator']}: {para['text'][:120]}...")

## Honest status of Phase 8 at this point

**Built in this phase**

| Piece | What it is |
|---|---|
| `app_service.py` | Facade over Phases 0-7 + upload pipeline + citation resolver + Review Tasks store |
| `ui.py` | 10-section Gradio workspace, teal design language from the mockup |
| `api.py` | 18 FastAPI endpoints over the *same* facade — UI and API cannot drift apart |
| `resilience.py` | Closed-loop failure recovery (detect → diagnose → select strategy → log) |

**Rubric gaps closed, and how**

- **Upload (was missing entirely):** a new `.docx` now goes through the real
  Phase 1 parser and Phase 3 indexer — the same path as the rest of the
  corpus, not a demo shortcut. You inspect the extracted clauses *before*
  paying for analysis.
- **Error handling (was open-loop):** failures are now classified first, and
  each class gets a **different** strategy — exponential backoff with jitter
  for transient errors, **model escalation** for schema failures, and
  **fail-fast** for quota/auth problems a retry could never fix. Every event
  is logged to `phase7_recovery_events.jsonl` and surfaced in the UI.
- **Workflow & state (was linear):** a cheap triage call now routes each
  document to only the specialists it plausibly needs, and routing
  **fails open** — a failed classification runs everything rather than
  silently skipping analysis. HITL is **risk-triggered**: findings escalate
  automatically on failed verification, low confidence, or CRITICAL severity.
- **Observability (was uniform `gpt-4o-mini`):** model tiering is now
  deliberate and documented in `config.py` — cheap triage, cheap default
  (justified by Phase 7's measured 94.6% citation validity), stronger model
  held in reserve for the case where a retry is known not to help.
- **Deployment (had no API):** FastAPI covers health, overview, library,
  assessments, analysis, citations, search, upload, reports (incl. file
  download), graph queries, tasks, activity and recovery events.
- **Citation provenance (your specific ask):** evidence is no longer an
  opaque chunk id. `resolve_citation()` opens the actual source document and
  returns the real paragraphs the chunk covers, **highlighting the exact
  paragraph that physically contains the quote** — using the same matching
  rule the verification step uses, so the highlight and the verdict agree.

**Verified, not assumed**

- 32 pytest tests pass (up from 11), including new suites for resilience,
  routing fail-open behavior, citation resolution, and the API.
- Every API endpoint was exercised against real artifacts with `TestClient`,
  including the 404/400 paths.
- The UI was launched and inspected live; the citation drill-down was
  confirmed to highlight `paragraph:7` of *MSA Sample 1.docx* as the real
  source of a finding's quote.
- A real bug was caught and fixed during this work: Gradio builds its
  stylesheet as a *constructed stylesheet*, where `@import` is silently
  ignored — the Google Fonts link had to move into `<head>` instead.

**Honest constraints that remain**

- This is a native Gradio layout, not a pixel clone of the mockup's
  hand-written HTML/CSS/JS. The information architecture, teal palette,
  Manrope/DM Sans typography and card/finding components are carried over;
  Gradio cannot render the mockup's custom JS components verbatim.
- `important_missed_risk_rate` still has no ground truth, and still reports
  `null` rather than a fabricated number.